In [3]:
pip install transformers datasets torch evaluate tokenizers transformers[torch]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.1 MB/s eta 0:00:0000:01
Note: you may need to restart the kernel to use updated packages.


In [38]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import torch.cuda.amp
from transformers import BertForTokenClassification, BertTokenizerFast
from datasets import Dataset
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score


# Load your data
data_path = "/kaggle/input/fashion/preprocessed_data.json"
df = pd.read_json(data_path, orient="records", lines=True)

# Filter the relevant columns and inspect the data
df = df[['product_category', 'title']]
categories = df["product_category"].unique().tolist()

# Function to assign labels based on product category matching in the title
def assign_labels(title, categories):
    tokens = title.split()  # Tokenize by splitting words
    labels = ["O"] * len(tokens)  # Default label is "O"

    for i, token in enumerate(tokens):
        for category in categories:
            category_words = category.split()  
            if token.lower() == category_words[0].lower():  
                if len(category_words) == 1:
                    labels[i] = "B-CLOTHING"
                else:
                    labels[i] = "B-CLOTHING"  
                    for j in range(1, len(category_words)):
                        if i + j < len(tokens) and tokens[i + j].lower() == category_words[j].lower():
                            labels[i + j] = "I-CLOTHING" 
    return tokens, labels

# Apply the function to the DataFrame to create tokens and labels
df['tokens_and_labels'] = df.apply(lambda row: assign_labels(row['title'], categories), axis=1)
df["tokens"] = df["tokens_and_labels"].apply(lambda x: x[0])
df["labels"] = df["tokens_and_labels"].apply(lambda x: x[1])

# Drop unnecessary columns
df.drop(columns=["tokens_and_labels"], inplace=True)


# Ensure the tokens and labels are lists
df["tokens"] = df["tokens"].apply(lambda x: x if isinstance(x, list) else [])
df["labels"] = df["labels"].apply(lambda x: x if isinstance(x, list) else [])

# Convert to a Hugging Face Dataset
dataset = Dataset.from_pandas(df[["tokens", "labels"]])

train_dataset, val_dataset = dataset.train_test_split(test_size=0.2).values()

# Initialize the tokenizer (TinyBERT-based tokenizer)
tokenizer = BertTokenizerFast.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")

# Tokenization function to prepare data for model
def tokenize_function(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        padding="max_length",  # Pad to max length
        is_split_into_words=True,
        max_length=128  # Adjust max length to your preference
    )
    label_map = {'O': 0, 'B-CLOTHING': 1, 'I-CLOTHING': 2}  # Label mapping

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = [
            -100 if word_id is None else label_map.get(label[word_id], -100)  # Default to -100 if no match
            for word_id in word_ids
        ]
        labels.append(aligned_labels)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

# Apply tokenization
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model = BertForTokenClassification.from_pretrained("huawei-noah/TinyBERT_General_4L_312D", num_labels=3)
tokenizer = BertTokenizerFast.from_pretrained("huawei-noah/TinyBERT_General_4L_312D")

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Set up DataLoader for train and validation sets
train_dataloader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=64)


optimizer = optim.Adam(model.parameters(), lr=1e-4)
loss_fn = torch.nn.CrossEntropyLoss()


# Initialize GradScaler for mixed precision training
scaler = torch.cuda.amp.GradScaler()

# Function to evaluate model on validation set
from sklearn.metrics import accuracy_score

# Function to evaluate model on validation set
def eval_model(model, val_dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    for batch in val_dataloader:
        data, targets = batch["input_ids"].to(device), batch["labels"].to(device)
        with torch.no_grad():
            output = model(data)
        
        # Get the predictions (output.logits) and convert to labels
        preds = output.logits.argmax(dim=-1)

        # Flatten the tensors and exclude padding tokens (those with label -100)
        preds = preds.flatten()
        targets = targets.flatten()

        # Collect the non-padding labels and predictions
        mask = targets != -100  # Mask padding tokens
        all_preds.extend(preds[mask].cpu().numpy())  # Only collect predictions for non-padding tokens
        all_labels.extend(targets[mask].cpu().numpy())  # Only collect true labels for non-padding tokens

    # Compute accuracy
    accuracy = accuracy_score(all_labels, all_preds)
    return accuracy


# Training loop with validation after each epoch
num_epochs = 10
best_val_accuracy = 0  # We will track the best validation accuracy
patience = 2
epochs_without_improvement = 0

for epoch in range(num_epochs):
    model.train()  # Set model to training mode
    for batch in train_dataloader:
        optimizer.zero_grad()

        # Move data to GPU
        data, targets = batch["input_ids"].to(device), batch["labels"].to(device)

        with torch.cuda.amp.autocast():  # Use mixed precision (FP16 where possible)
            output = model(data)
            logits = output.logits  # Extract logits
            loss = loss_fn(logits.view(-1, 3), targets.view(-1))  # Compute loss

        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    # After each epoch, evaluate on validation set
    val_accuracy = eval_model(model, val_dataloader)
    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {loss.item()}, Validation Accuracy: {val_accuracy}")

    # Early stopping check (if validation accuracy does not improve)
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"Early stopping after {epoch+1} epochs. No improvement in validation accuracy.")
        break
# Save the model's state_dict after training
torch.save(model.state_dict(), '/kaggle/working/clothing-ner-model.pth')

# Optionally, you can also save the tokenizer
tokenizer.save_pretrained('/kaggle/working/clothing-ner-tokenizer')



Map:   0%|          | 0/22471 [00:00<?, ? examples/s]

Map:   0%|          | 0/5618 [00:00<?, ? examples/s]

Some weights of BertForTokenClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-38-21323394df05>:106: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()
<ipython-input-38-21323394df05>:153: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():  # Use mixed precision (FP16 where possible)


Epoch 1/10, Loss: 0.003984533250331879, Validation Accuracy: 0.9989215198274431


<ipython-input-38-21323394df05>:153: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():  # Use mixed precision (FP16 where possible)


...

<ipython-input-38-21323394df05>:153: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():  # Use mixed precision (FP16 where possible)


Epoch 8/10, Loss: 0.0002560340508352965, Validation Accuracy: 0.9993570598971295


<ipython-input-38-21323394df05>:153: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():  # Use mixed precision (FP16 where possible)


Epoch 9/10, Loss: 0.0001657201355556026, Validation Accuracy: 0.9997718599634976
Early stopping after 9 epochs. No improvement in validation accuracy.


('/kaggle/working/clothing-ner-tokenizer/tokenizer_config.json',
 '/kaggle/working/clothing-ner-tokenizer/special_tokens_map.json',
 '/kaggle/working/clothing-ner-tokenizer/vocab.txt',
 '/kaggle/working/clothing-ner-tokenizer/added_tokens.json',
 '/kaggle/working/clothing-ner-tokenizer/tokenizer.json')

In [41]:
import torch
from transformers import BertTokenizerFast
from torch.utils.data import DataLoader
from transformers import BertForTokenClassification

# Reload the tokenizer
tokenizer = BertTokenizerFast.from_pretrained('/kaggle/working/clothing-ner-tokenizer')

# Reload the model architecture (TinyBERT)
model = BertForTokenClassification.from_pretrained('huawei-noah/TinyBERT_General_4L_312D', num_labels=3)

# Load the saved state_dict
model.load_state_dict(torch.load('/kaggle/working/clothing-ner-model.pth'))

# Move the model to the appropriate device (GPU if available, else CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Sample data (use real data in your case)
sample_data = [
    "Red cotton T-shirt with graphic print",
    "Slim-fit denim jeans for men",
    "Black leather jacket with hood",
    "Cozy wool sweater in navy blue",
    "Sporty running shoes for all-day comfort"
]

# Tokenize the input text
def tokenize_sample_text(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        is_split_into_words=False,
        return_tensors="pt"
    )

# Apply the tokenization
inputs = tokenize_sample_text(sample_data)

# Make predictions
model.eval()
with torch.no_grad():
    outputs = model(**inputs.to(device))
    logits = outputs.logits

# Get predictions
predictions = torch.argmax(logits, dim=-1)

# Define the label map
label_map = {0: 'O', 1: 'B-CLOTHING', 2: 'I-CLOTHING'}

# Decode predictions for each sample
for idx, text in enumerate(sample_data):
    print(f"Text: {text}")
    predicted_labels = [label_map[pred.item()] for pred in predictions[idx]]
    print(f"Predicted Labels: {predicted_labels}\n")



Some weights of BertForTokenClassification were not initialized from the model checkpoint at huawei-noah/TinyBERT_General_4L_312D and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-41-4149ef858e9a>:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_

Text: Red cotton T-shirt with graphic print
Predicted Labels: ['B-CLOTHING', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Text: Slim-fit denim jeans for men
Predicted Labels: ['B-CLOTHING', 'O', 'O', 'O', 'O', 'I-CLOTHING', 'O', 'O', 'O', 'O', 'O']

Text: Black leather jacket with hood
Predicted Labels: ['B-CLOTHING', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Text: Cozy wool sweater in navy blue
Predicted Labels: ['B-CLOTHING', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Text: Sporty running shoes for all-day comfort
Predicted Labels: ['B-CLOTHING', 'O', 'O', 'O', 'I-CLOTHING', 'O', 'O', 'O', 'O', 'O', 'O']



In [52]:
def decode_predictions(predictions, inputs, label_map):
    word_ids = inputs.word_ids(batch_index=0)  # For the first sentence
    merged_predictions = []
    current_word_id = None

    for token_id, word_id in zip(predictions[0], word_ids):
        label = label_map[token_id.item()]

        if word_id != current_word_id:
            # New word: append its label
            if current_word_id is not None:  # Don't append for the very first token
                merged_predictions.append(label)
        else:
            # If it's the same word (same word_id), continue with the previous label
            if merged_predictions:
                if 'B-CLOTHING' in merged_predictions[-1]:
                    merged_predictions[-1] = 'I-CLOTHING'  # Continue with I-CLOTHING
                else:
                    merged_predictions[-1] = label  # Otherwise, keep the label

        current_word_id = word_id  # Update current word_id

    # Ensure to append the last label
    if len(predictions[0]) > 0 and (current_word_id is not None):
        merged_predictions.append(label)

    return merged_predictions



In [53]:
# this function needs to be fixed
# Apply decoding for merged predictions
merged_predictions = decode_predictions(predictions, inputs, label_map)

# Show results
for idx, text in enumerate(sample_data):
    print(f"Text: {text}")
    print(f"Merged Predicted Labels: {merged_predictions}\n")


Text: Red cotton T-shirt with graphic print
Merged Predicted Labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Text: Slim-fit denim jeans for men
Merged Predicted Labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Text: Black leather jacket with hood
Merged Predicted Labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Text: Cozy wool sweater in navy blue
Merged Predicted Labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

Text: Sporty running shoes for all-day comfort
Merged Predicted Labels: ['O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']

